# CRISPR host species assignments: 0 vs 1 mismatch

Compare the number of species-level CRISPR host assignments when requiring exact spacer matches (`N mismatches == 0`) versus allowing one mismatch (`N mismatches <= 1`), using the consensus logic from `toolkit/bin/uhvdb_crisprhost.py`.

In [1]:
### Load packages
import polars as pl

In [2]:
### Paths
CRISPR_TSV = '../figure_1/uhgv_hq_hc_results/uhvdb_2026-03-23/uhvdb_crispr.tsv.gz'
SPECIES_INFO = '../figure_1/uhgv_hq_hc_results/uhvdb_2026-03-23/uhvdb_species_info.tsv.gz'
MIN_AGREEMENT = 0.7

### Assign hosts with consensus threshold

Mirrors `uhvdb_crisprhost.py`: for each virus (`uhvdb_id`), count spacer connections per species, take the top species, and keep assignments with agreement ≥ 0.7.

In [3]:
def assign_species_hosts(crispr_tsv, max_mismatches=1, min_agreement=0.7):
    """Return species-level CRISPR host assignments for a mismatch threshold."""
    return (
        pl.scan_csv(crispr_tsv, separator='\t')
            .select(['uhvdb_id', 'species', 'N mismatches'])
            .filter(pl.col('N mismatches') <= max_mismatches)
            .group_by(['uhvdb_id', 'species'])
            .agg(pl.len().alias('connections'))
            .group_by('uhvdb_id')
            .agg([
                pl.col('connections').sum().alias('total_connections'),
                pl.col('connections').max().alias('max_connections'),
                pl.col('species').sort_by('connections', descending=True).first().alias('top_taxonomy'),
            ])
            .with_columns([
                (pl.col('max_connections') / pl.col('total_connections')).alias('agreement'),
                pl.lit('species').alias('rank'),
            ])
            .filter(pl.col('agreement') >= min_agreement)
            .collect(engine='streaming')
    )

In [4]:
### Species assignments at 0 vs 1 allowed mismatch
hosts_0mm = assign_species_hosts(CRISPR_TSV, max_mismatches=0, min_agreement=MIN_AGREEMENT)
hosts_1mm = assign_species_hosts(CRISPR_TSV, max_mismatches=1, min_agreement=MIN_AGREEMENT)

n_species_0mm = hosts_0mm['uhvdb_id'].n_unique()
n_species_1mm = hosts_1mm['uhvdb_id'].n_unique()

summary = pl.DataFrame({
    'max_mismatches': [0, 1],
    'n_species_assignments': [n_species_0mm, n_species_1mm],
})

print(f'Species assignments with 0 mismatches allowed: {n_species_0mm}')
print(f'Species assignments with 1 mismatch allowed:  {n_species_1mm}')
print(f'Difference (1mm - 0mm): {n_species_1mm - n_species_0mm}')
summary

Species assignments with 0 mismatches allowed: 60479
Species assignments with 1 mismatch allowed:  57307
Difference (1mm - 0mm): -3172


max_mismatches,n_species_assignments
i64,i64
0,60479
1,57307


In [5]:
### Overlap between 0-mismatch and 1-mismatch species assignments
shared = (
    hosts_0mm.select(['uhvdb_id', 'top_taxonomy'])
        .join(
            hosts_1mm.select(['uhvdb_id', 'top_taxonomy']),
            on='uhvdb_id',
            how='inner',
            suffix='_1mm',
        )
)
same_species = shared.filter(pl.col('top_taxonomy') == pl.col('top_taxonomy_1mm')).height

print(f'Viruses with species assignment under both thresholds: {shared.height}')
print(f'Of those, same top species under both thresholds: {same_species}')
print(f'Only in 0-mismatch set: {n_species_0mm - shared.height}')
print(f'Only in 1-mismatch set: {n_species_1mm - shared.height}')

Viruses with species assignment under both thresholds: 49127
Of those, same top species under both thresholds: 48155
Only in 0-mismatch set: 11352
Only in 1-mismatch set: 8180


### Percent of species clusters with a species-level host prediction

A species cluster is counted if any genome in the cluster has a consensus species-level CRISPR host assignment (agreement ≥ 0.7).

In [ ]:
### Load species clusters and compute coverage
species_info = (
    pl.read_csv(SPECIES_INFO, separator='\t')
        .select(['uhvdb_id', 'cluster_id', 'votu_rep'])
)
n_clusters = species_info['cluster_id'].n_unique()

hosts_either = pl.concat([
    hosts_0mm.select('uhvdb_id'),
    hosts_1mm.select('uhvdb_id'),
]).unique()

cluster_coverage = []
for label, max_mm, hosts in [
    ('0 mismatches', 0, hosts_0mm),
    ('1 mismatch', 1, hosts_1mm),
    ('either 0 or 1', None, hosts_either),
]:
    n_with_pred = (
        species_info
            .join(hosts.select('uhvdb_id'), on='uhvdb_id', how='inner')
            ['cluster_id'].n_unique()
    )
    cluster_coverage.append({
        'mismatch_threshold': label,
        'max_mismatches': max_mm,
        'n_clusters_with_species_pred': n_with_pred,
        'n_species_clusters': n_clusters,
        'percent_clusters': 100 * n_with_pred / n_clusters,
    })

cluster_summary = pl.DataFrame(cluster_coverage)
print(f'Total species clusters: {n_clusters}')
for row in cluster_summary.iter_rows(named=True):
    print(
        f"{row['mismatch_threshold']}: "
        f"{row['n_clusters_with_species_pred']}/{row['n_species_clusters']} "
        f"({row['percent_clusters']:.2f}%)"
    )
cluster_summary

Total species clusters: 53595
0 mismatches: 25073/53595 (46.78%)
1 mismatch: 23626/53595 (44.08%)
either 0 or 1: 28597/53595 (53.36%)


mismatch_threshold,max_mismatches,n_clusters_with_species_pred,n_species_clusters,percent_clusters
str,i64,i64,i64,f64
"""0 mismatches""",0,25073,53595,46.782349
"""1 mismatch""",1,23626,53595,44.08247
"""either 0 or 1""",null,28597,53595,53.357589
